# Train Slang Normalizer

This notebook is an organized, runnable conversion of `train_normalizer.py`.

It contains parameter cells, helper functions, training pipeline, and a final execution cell that trains and saves the model payload.

In [1]:
# Notebook parameters and paths
csv_path = "../complete_slang_normalization_dataset.csv"
output_path = "model.joblib"
extra_english = None  # e.g., "data/english_lines.txt"
extra_filipino = None  # e.g., "data/filipino_lines.txt"
use_language_tags = True
valid_split = 0.1
seed = 13

# You can modify the values above and re-run the cells below to train with different settings.

In [ ]:
# Imports and reproducibility setup
import json
import random
from collections import Counter
from pathlib import Path

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# set seed for reproducible splits/shuffling
random.seed(seed)

In [3]:
# Load text line utilities
from pathlib import Path

def load_lines(path):
    lines = []
    if not path:
        return lines
    p = Path(path)
    if not p.exists():
        return lines
    with p.open("r", encoding="utf-8") as handle:
        for raw in handle:
            text = raw.strip()
            if text:
                lines.append(text)
    return lines

In [4]:
# Language tagging helpers
def add_language_tag(text, language, use_language_tags):
    if not use_language_tags:
        return text
    tag = "__lang_en__" if language == "english" else "__lang_fil__"
    return f"{tag} {text}"

In [5]:
# Build training examples from CSV
def build_examples(df, use_language_tags):
    inputs = []
    outputs = []
    for _, row in df.iterrows():
        slang = str(row["slang"]).strip()
        normalized = str(row["normalized"]).strip()
        language = str(row.get("language", "english")).strip().lower()
        if not slang or not normalized:
            continue
        inputs.append(add_language_tag(slang, language, use_language_tags))
        outputs.append(normalized)
    return inputs, outputs

In [6]:
# Add identity examples from extra corpora
def add_identity_examples(inputs, outputs, lines, language, use_language_tags):
    for text in lines:
        tagged = add_language_tag(text, language, use_language_tags)
        inputs.append(tagged)
        outputs.append(text)

In [ ]:
# Train/validation split and model pipeline

def train_model(inputs, outputs, valid_split, random_seed):
    if valid_split > 0:
        label_counts = Counter(outputs)
        min_count = min(label_counts.values()) if label_counts else 0
        stratify_labels = outputs if min_count >= 2 else None
        x_train, x_valid, y_train, y_valid = train_test_split(
            inputs,
            outputs,
            test_size=valid_split,
            random_state=random_seed,
            stratify=stratify_labels,
        )
    else:
        x_train, y_train = inputs, outputs
        x_valid, y_valid = [], []

    pipeline = Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=1,
                ),
            ),
            (
                "clf",
                LogisticRegression(max_iter=2000, solver="lbfgs"),
            ),
        ]
    )

    pipeline.fit(x_train, y_train)

    metrics = {}
    if x_valid:
        preds = pipeline.predict(x_valid)

        # Accuracy
        metrics["valid_accuracy"] = accuracy_score(y_valid, preds)

        # BLEU (try nltk then sacrebleu; set None if unavailable)
        try:
            from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
            references = [[r.split()] for r in y_valid]
            candidates = [p.split() for p in preds]
            bleu = corpus_bleu(references, candidates, smoothing_function=SmoothingFunction().method1)
            metrics["valid_bleu"] = float(bleu)
        except Exception:
            try:
                import sacrebleu
                refs = [r for r in y_valid]
                bleu_score = sacrebleu.corpus_bleu(preds, [refs]).score / 100.0
                metrics["valid_bleu"] = float(bleu_score)
            except Exception:
                metrics["valid_bleu"] = None

        # F1 (weighted) and classification report
        try:
            f1_weighted = f1_score(y_valid, preds, average="weighted", zero_division=0)
            metrics["valid_f1_weighted"] = float(f1_weighted)
            metrics["classification_report"] = classification_report(y_valid, preds, zero_division=0, output_dict=True)
        except Exception:
            metrics["valid_f1_weighted"] = None
            metrics["classification_report"] = None

        # Confusion matrix (labels and matrix)
        try:
            labels = sorted(list(set(y_valid) | set(preds)))
            cm = confusion_matrix(y_valid, preds, labels=labels).tolist()
            metrics["confusion_matrix"] = {"labels": labels, "matrix": cm}
        except Exception:
            metrics["confusion_matrix"] = None

    return pipeline, metrics

In [8]:
# End-to-end training execution

# Load dataset
import os
csv_p = Path(csv_path)
if not csv_p.exists():
    # try workspace root
    csv_p = Path(os.getcwd()) / csv_path

df = pd.read_csv(csv_p)

# Build examples
inputs, outputs = build_examples(df, use_language_tags)

# Add optional identity examples
if extra_english:
    eng_lines = load_lines(extra_english)
    add_identity_examples(inputs, outputs, eng_lines, "english", use_language_tags)

if extra_filipino:
    fil_lines = load_lines(extra_filipino)
    add_identity_examples(inputs, outputs, fil_lines, "filipino", use_language_tags)

# Shuffle and prepare
combined = list(zip(inputs, outputs))
random.shuffle(combined)
inputs, outputs = zip(*combined)

# Train
model, metrics = train_model(list(inputs), list(outputs), valid_split, seed)

# Prepare payload
payload = {
    "pipeline": model,
    "use_language_tags": use_language_tags,
    "metadata": {
        "valid_split": valid_split,
        "metrics": metrics,
        "label_count": len(set(outputs)),
    },
}

# Serialize
joblib.dump(payload, output_path)

# Display metadata
print(json.dumps(payload["metadata"], indent=2))

{
  "valid_split": 0.1,
  "metrics": {
    "valid_accuracy": 0.3736995422388681
  },
  "label_count": 86
}


Next steps: open the saved model (`model.joblib`) and use `pipeline` to normalize text in your extensio